# Analysis 8: Midterm Scores, Activity Patterns, and Visual Findings

This notebook is the visual version of the analysis. It uses the saved outputs in `analysis8_outputs` and is written for an instructor who wants to understand how students used the Runestone ebook and what parts of that usage were most closely associated with midterm outcomes.

## Big takeaways

- **Midterm 2 was harder almost everywhere.** Across all semesters with both exams, the average `mid2 - mid1` shift was **-8.3 percentage points**, and the largest drop was **F22 (-13.2)**.
- **Parsons first-try quality was the strongest single signal.** Students in the top quartile of `Parsons mean first score` scored about **17.5 points higher** on the midterm than students in the bottom quartile of the same semester-midterm cohort.
- **Breadth beat raw clicking for concept checks.** `conceptcheck__unique_problems` was positive (**0.07**), while `conceptcheck__total_events` was slightly negative (**-0.09**).
- **Big practice gains were not automatically “good news.”** `parsons__mean_score_gain` was strongly negative (**-0.44**), which suggests large gains often reflected weaker starting points rather than stronger final mastery.
- **Behavior changes before midterm 2 mattered, but less than baseline quality.** The strongest change feature was `Parsons: Total Events  Change` at **0.13**, which is real but much smaller than the top baseline-quality signals.
- **Efficiency seems to matter more than sheer effort.** Students with high Parsons efficiency scored much better than low-efficiency students, even when both groups showed high effort.
- **Timed practice midterms added a useful readiness signal.** Finishing a timed practice exam before the real exam was associated with about **6.2 more real-exam points** within the same cohort, and timed-practice quality itself had a positive within-cohort relationship of about **0.27**.


## High-Level Pipeline

1. **Raw Runestone log events (19,805,383)**: All anonymized ebook log rows across reading, lecture, practice, and exam activity.
2. **Timed actual midterm attempts (2,906)**: Student exam windows inferred from timed start and finish markers in the log.
3. **Student-exam problem scores (90,048)**: Observed exam problem rows scored from clear correctness signals or high-confidence inferred keys.
4. **Student midterm score rows (2,314)**: One reconstructed midterm row per student, semester, and exam.
5. **Student pre-exam feature rows (2,314)**: Pre-exam activity summaries used to compare students within the same semester-midterm cohort.

This section is here to make the later plots more trustworthy. The report does not jump straight from raw clicks to conclusions. It first identifies real exam windows, reconstructs student-level midterm scores, then measures pre-exam behavior and compares students within the same semester-midterm cohorts.

## How To Read These Figures

- Every correlation is computed **within semester-midterm cohorts** first, then combined, so the plots are not just picking up that one semester was easier than another.
- The scatter plots use **percentile ranks within each cohort** on both axes. A point near `(80, 80)` means a student is around the 80th percentile on both the feature and the exam score within their own cohort.
- Score reconstruction comes from the `analysis7` exam windows plus event-level correctness signals. Ambiguous `fillb`, `dragndrop`, `clickable`, and `shortanswer` rows are shown explicitly in the limitations section instead of being force-scored.
- The timed practice-midterm analysis is intentionally conservative. It uses only **timed practice windows that finished before the real exam**, and its score is based on directly scoreable practice items like multiple choice, Parsons checks, and unittest-backed coding items.


## Plain-Language Glossary

- **Parsons puzzle**: a programming problem where students arrange mixed-up code blocks into the correct order. In the logs, `parsonsMove` records moves and `parsons` records graded checks.
- **Parsons first-try quality**: for each Parsons problem a student worked before the exam, I looked at the **first time the student checked their answer**. If that first check was correct, that problem contributes a high value; if it was incorrect, it contributes a low value. The student’s feature is the average across all of those problems. In classroom terms, this is a measure of whether students can set up a correct solution quickly rather than after lots of trial-and-error.
- **ActiveCode**: executable code cells in the ebook. The feature `ActiveCode mean best score` uses the best unit-test score the student reached on each coding problem before the exam.
- **Concept checks**: the `selectquestion` family in the logs. These appear to be the course’s regular non-exam multiple-choice concept questions, including the kinds of questions used in lecture and reading checks.
- **Total events**: every logged interaction, such as a click, move, run, or answer submission. This is a volume measure.
- **Unique problems**: the number of distinct problems a student touched. This is a breadth measure.
- **Active days**: the number of different calendar days on which the student used that activity family before the exam. This is a spacing measure.
- **Mean score gain**: the student’s last observed score on a practice problem minus their first observed score on that same problem, averaged across problems. A negative correlation here does **not** mean improvement is bad. It usually means students who began weaker had more room to improve.
- **Pre-exam**: all activity that happened before the start of the relevant midterm window. `mid1` features use behavior before midterm 1, and `mid2` features use behavior before midterm 2.


## Teacher Interpretation Notes

- These visuals are best used to identify **patterns worth acting on**, not to claim causation.
- A **positive correlation** means students who show more of that pattern also tend to earn stronger midterm scores.
- A **negative correlation** often means the feature is acting as a marker of struggle. For example, many Parsons checks or large score gains can mean a student needed a lot of retries before they understood the problem.
- For teaching decisions, the safest interpretation is: if a feature reflects **clean early understanding** or **broad, consistent practice**, it is a promising signal. If a feature reflects **heavy rework**, it may help flag students who need support earlier.


## Connecting The Dots

- **Why does “did the activity or not” barely matter?** Because in this course, the major activity families were already close to universal. Once nearly everyone has touched Parsons, ActiveCode, concept checks, and multiple choice, the interesting differences are about *quality*, *breadth*, and *consistency*.
- **Why are quality metrics stronger than raw counts?** A student who gets practice items mostly right on the first graded check is probably showing fluency. A student with a huge number of moves, checks, or events may instead be showing confusion, persistence, or both. Those are not the same instructional signal.
- **Why can score gain be negative even if practice helps?** Because gain is strongly shaped by starting point. In this dataset, students in the lowest Parsons first-try quartile improved by about **0.71** on average, while students in the highest quartile improved by only **0.39**. But the lower-starting group still averaged only **61.9%** on the midterm versus **79.2%** for the highest-starting group. That means improvement is real, but it often reflects students climbing out of a deeper hole rather than overtaking stronger peers.
- **Why might total concept-check events be weak or slightly negative?** A likely reason is that raw clicks mix together productive engagement and repeated revisiting caused by uncertainty. In contrast, the number of distinct concept-check problems and the number of days students engaged with them are more clearly positive, which fits the idea that broad, spaced exposure is healthier than repeated clicking on the same material.
- **What about cramming?** The timing story is not as simple as “strong students never cram.” In fact, strong students also do a sizable share of their activity near the exam. The difference seems to be that final-week activity by itself is not enough to explain performance; students still separate mostly on quality and efficiency.
- **Why is midterm 2 lower than midterm 1 almost everywhere?** The log data alone cannot prove the cause. Plausible explanations include more difficult later content, a harder exam, cumulative course load, or a broader skill mix on the second exam. The consistent cross-semester pattern makes it worth discussing as a structural course issue, not just a one-term anomaly.


In [ ]:
from pathlib import Path
import pandas as pd

base_dir = Path.cwd()
out_dir = base_dir / "analysis8_outputs"

scores = pd.read_parquet(out_dir / "student_midterm_scores.parquet")
features = pd.read_parquet(out_dir / "student_midterm_activity_features.parquet")
semester_summary = pd.read_csv(out_dir / "semester_midterm_score_summary.csv")
activity_corr = pd.read_csv(out_dir / "activity_correlations.csv")
change_corr = pd.read_csv(out_dir / "activity_change_correlations.csv")
unsupported = pd.read_csv(out_dir / "unsupported_or_ambiguous_exam_items.csv")


## Who Actually Used Each Activity Type?

This figure answers a basic course-design question first: before each midterm, what percentage of students had **any** interaction with each major activity family? The answer is: almost everyone. For example, Parsons usage is already about **99.7%** before `mid1`, and concept checks are about **99.7%**. That means simple participation is not the key differentiator in this course. Students generally encountered all of these tools. The more useful question for an instructor is **how** they used them: quickly or slowly, broadly or narrowly, on one day or over many days, with immediate correctness or repeated rework. Teacher takeaway: because exposure is nearly universal, interventions should focus less on getting students to click once and more on helping them practice successfully and consistently.

![Who Actually Used Each Activity Type?](analysis8_outputs/figures/10_participation_rates.png)

## How Much Practice Did A Typical Student Do?

This chart shows the median number of distinct problems a student had touched before each exam. It gives a sense of what “normal” exposure looked like in this course. A typical student had worked around **52 Parsons problems** before `mid1` and **89** before `mid2`. For ActiveCode, the median rose from about **144** to **208**. The main point is that students had substantial contact with these activity types well before the exams. Teacher takeaway: if an instructor wants to improve outcomes, the strongest leverage may be improving the *quality* of these interactions rather than simply adding more instances of the same activity.

![How Much Practice Did A Typical Student Do?](analysis8_outputs/figures/11_typical_exposure.png)

## Semester Overview

This figure has two panels. The top panel shows the average reconstructed midterm score in each semester, split into `mid1` and `mid2`. The bottom panel shows how much of each exam could be scored from observable item-level signals. The main story is that `mid2` sits below `mid1` in every semester, while scoring coverage stays high enough that this pattern is unlikely to be a data artifact. For a teacher, this suggests the second exam may consistently be harder, later material may be more challenging, or cumulative fatigue may be setting in by that point of the semester. Teacher takeaway: if this pattern matches classroom experience, the instructor may want to inspect the content, pacing, and support structures leading into the second midterm.

![Semester Overview](analysis8_outputs/figures/01_semester_overview.png)

## Score Distributions By Semester

These boxen plots show the spread of student scores within each semester and midterm, not just the average. This helps separate two ideas: whether one exam was lower on average, and whether it was also more spread out. Several `mid2` cohorts shift downward as a whole rather than just adding a few low outliers, which supports the idea that the second midterm was broadly tougher. In other words, this does not look like a story driven by a tiny subgroup alone. Teacher takeaway: if an exam shift affects the whole distribution, the response may need to be course-wide, such as earlier review, more scaffolding, or adjusted expectations for the later unit.

![Score Distributions By Semester](analysis8_outputs/figures/02_score_distributions.png)

## Strongest Activity Signals

This lollipop chart ranks the strongest positive and negative relationships between pre-exam behavior and midterm performance. Positive values mean the feature tends to go with stronger midterm scores; negative values mean the feature tends to go with weaker scores. The standout result is Parsons quality: first-try and average Parsons correctness dominate the chart. The surprising negative result is that large score gains on practice are often a weakness signal, because students who started much lower had more room to improve. This is one of the most important interpretation points for the whole notebook: high effort is not always the same thing as high preparedness. Teacher takeaway: repeated retries can be used as an early-warning signal for struggle, while clean early success can be used as a signal of readiness.

![Strongest Activity Signals](analysis8_outputs/figures/03_activity_correlation_lollipop.png)

## Quality Versus Volume Heatmap

This heatmap groups the correlations by activity family and metric type. Read across a row to compare one metric, or down a column to see the overall pattern for a family. Warm cells are positive, cool cells are negative. The main pattern is very consistent: quality metrics such as `mean first score`, `mean best score`, and `average event score` are more informative than raw event counts. Concept checks are especially interesting because `unique problems` and `active days` help, while simple `total events` do not. In practical terms, touching more concept-check questions across more days seems healthier than clicking the same question family many times in a concentrated burst. Teacher takeaway: this argues for distributed practice and broad coverage rather than just more activity volume.

![Quality Versus Volume Heatmap](analysis8_outputs/figures/04_quality_vs_volume_heatmap.png)

## Parsons First-Try Quality Versus Midterm Outcome

This scatter plot puts both the Parsons feature and the exam score into within-cohort percentiles, so each dot compares a student against peers from the same semester and same midterm. The upward slope is steep: moving from the bottom quartile to the top quartile of Parsons first-try quality is associated with about **17.5 extra midterm points**. That is one of the strongest effects anywhere in the notebook. To make this concrete for a teacher: students who can assemble a reasonable Parsons solution and get it correct on the first graded check tend to be much more exam-ready than students who need many checks before arriving at the answer. Teacher takeaway: Parsons puzzles may be especially useful not just as practice, but as a diagnostic. Students who repeatedly miss the first check could be flagged for support before the exam.

![Parsons First-Try Quality Versus Midterm Outcome](analysis8_outputs/figures/05_parsons_first_score_scatter.png)

## Effort Versus Efficiency

This plot compares two different ideas that are easy to confuse. The horizontal axis is **effort**: how much Parsons activity a student generated relative to classmates in the same cohort. The vertical axis is **efficiency**: how strong that student’s Parsons first-try quality was relative to classmates. The color shows the student’s midterm score percentile. The key pattern is that high-efficiency students tend to do well whether their effort is low or high, while low-efficiency students tend to struggle even when their effort is high. Teacher takeaway: a lot of activity is not automatically a sign of understanding. A student can be working very hard and still need help. This makes Parsons logs useful for distinguishing productive practice from repeated struggle.

![Effort Versus Efficiency](analysis8_outputs/figures/13_effort_vs_efficiency.png)

## Student Behavior Profiles

This bar chart turns the effort-efficiency map into four simple profiles. The strongest profile is **High effort, high efficiency**, with an average midterm score around **77.2%**. The weakest profile is **Low effort, low efficiency**, at about **65.0%**. The especially interesting comparison is between **high effort, high efficiency** and **high effort, low efficiency**: effort alone does not close the gap. Teacher takeaway: if an instructor wants an early warning flag, high-effort students with low efficiency may be one of the most important groups to catch, because they are engaged but still not converting effort into understanding.

![Student Behavior Profiles](analysis8_outputs/figures/14_student_profiles.png)

## ActiveCode Best Practice Quality Versus Midterm Outcome

This plot repeats the same percentile idea for `ActiveCode mean best score`. The relationship is still clearly positive, but not as dominant as Parsons. Students in the top quartile of this feature outscored bottom-quartile students by about **7.8 points**, which is still a meaningful gap. Since ActiveCode problems are executable and testable code, this result is consistent with the idea that students who can get coding practice to pass unit tests before the exam are more likely to perform well on the exam too. Teacher takeaway: coding practice quality matters, but the stronger Parsons signal suggests that code-tracing and code-assembly fluency may be especially important in this course.

![ActiveCode Best Practice Quality Versus Midterm Outcome](analysis8_outputs/figures/06_activecode_best_score_scatter.png)

## Timed Practice Midterms: Participation And Completion

This figure introduces a separate but very practical question for instructors: what happened when students used the course’s timed practice midterms before the real exam? The left panel compares actual midterm scores for three groups: students who did no timed practice, students who opened a timed practice exam but did not finish one before the real exam, and students who finished one. The strongest group is the finishers. On `mid1`, students who finished a timed practice exam averaged about **77.1%** on the real exam, compared with **71.7%** for students who did none. On `mid2`, the same comparison is about **68.7%** versus **65.5%**. The right panel shows that timed practice usage varied by semester, which matters because it reminds us this is an opportunity signal, not something every cohort received equally. Teacher takeaway: simply opening a practice exam is not the same as working through it. Completion appears to be the more meaningful readiness marker.

![Timed Practice Midterms: Participation And Completion](analysis8_outputs/figures/17_practice_midterm_groups.png)

## Timed Practice Midterms: Quality As A Readiness Check

This figure goes one step further. Among students with a finished timed practice exam, it asks whether stronger performance on the directly scoreable practice items also lined up with stronger real-exam performance. The left scatter uses within-cohort percentiles, and the relationship is clearly positive at about **0.27**. The right panel makes that easier to read: students in the **top timed-practice quality quartile** scored about **8.8 points** higher on the real exam than students in the bottom quartile of the same cohort. This practice score is conservative: it only uses directly scoreable items inside the timed practice windows, and it is based on roughly **1695 finished timed practice attempts** overall. Teacher takeaway: practice midterms appear most useful as a diagnostic when students actually complete them and when instructors pay attention to how well students do, not just whether the practice link was opened.

![Timed Practice Midterms: Quality As A Readiness Check](analysis8_outputs/figures/18_practice_midterm_quality.png)

## Cramming Versus Consistency

This figure checks a common hypothesis directly. The left panel shows how top and bottom exam quartiles distributed their activity over the last 28 days before the exam. The right panel shows how many separate days they were active in that window. The result is more nuanced than a simple anti-cramming story: top students still did a lot of work in the last week, averaging about **44.4%** of their last-28-day activity there, compared with **37.2%** for the bottom quartile. Their average number of active days was also only slightly higher, about **11.3** versus **10.9**. Teacher takeaway: last-week review seems normal for everyone. Timing by itself is not the main separator here; practice quality remains the stronger signal.

![Cramming Versus Consistency](analysis8_outputs/figures/15_cramming_vs_consistency.png)

## What Changed From Midterm 1 To Midterm 2?

This chart looks only at students who have both midterms in the same semester and asks which behavior changes line up with score changes. The bars are noticeably smaller than the baseline-quality plots above, which is an important result by itself: what students already know going into the exam matters more than short-term behavior changes. Still, the best change signals lean toward increasing Parsons engagement before `mid2`. Teacher takeaway: last-minute changes in behavior can help somewhat, but the bigger instructional opportunity may be building stronger habits and understanding earlier in the term rather than relying on short recovery windows before the second exam.

![What Changed From Midterm 1 To Midterm 2?](analysis8_outputs/figures/07_change_correlation_bars.png)

## Parsons Growth Groups Before Midterm 2

This grouped bar chart breaks students into lower- versus upper-baseline Parsons groups and then compares low, middle, and high growth in total Parsons activity before `mid2`. Everyone still tends to drop from `mid1` to `mid2`, but higher Parsons growth usually softens that drop. In the lower-baseline group, moving from low to high Parsons growth changed the average `mid2 - mid1` drop by about **3.2 points**. That is one of the clearest change-based signals in the notebook, even though it is still smaller than the baseline-quality effects. Teacher takeaway: Parsons practice appears especially promising as a support tool for students who are not already strong. It may not erase the gap, but it may help reduce the size of the decline on the second exam.

![Parsons Growth Groups Before Midterm 2](analysis8_outputs/figures/08_parsons_growth_groups.png)

## Learning Progression

This figure compares first observed and last observed practice scores for the lowest and highest exam quartiles across several activity families. The big pattern is that both groups improve, but the stronger exam group usually starts higher and stays higher. For Parsons, for example, the lowest exam quartile starts around **0.27** and ends around **0.91**, while the highest exam quartile starts higher at about **0.48** and ends higher at about **0.96**. Teacher takeaway: the issue is not that struggling students fail to improve at all. Many do improve. The issue is that early gaps remain meaningful by exam time, which argues for support that begins earlier rather than only after students have already fallen behind.

![Learning Progression](analysis8_outputs/figures/16_learning_progression.png)

## Why Improvement Margin Can Mislead

This figure directly addresses one of the main interpretation traps in the data. Students in the **lowest** Parsons first-try quartile improved the most on Parsons practice, averaging about **0.71** points of gain, while students in the **highest** quartile improved less, around **0.39**. But the lower-starting group still ended with lower exam scores overall. This is why `Parsons mean score gain` shows up as a negative correlation: it is often measuring *how far behind a student started*, not whether practice was useless. Teacher takeaway: a large improvement margin can actually be a sign that a student needed substantial recovery. That is useful information, but it should be interpreted as a support signal rather than a failure of the activity.

![Why Improvement Margin Can Mislead](analysis8_outputs/figures/12_parsons_baseline_story.png)

## Scoring Limitations

This final figure shows which exam item families were left unscored because the logs were ambiguous or did not contain a stable correctness signal. Most of the unresolved rows are `fillb`, not multiple choice, Parsons, or ActiveCode. That means the main headline relationships are being driven by the activity families with the cleanest signals, while the murkier item types are kept separate instead of being guessed. Teacher takeaway: you can place more confidence in the trends involving Parsons, ActiveCode, and multiple-choice practice than in anything tied to the noisier item types.

![Scoring Limitations](analysis8_outputs/figures/09_scoring_limitations.png)